# Hebrew → subtitles (Whisper large-v3)

Generates accurate `.srt` subtitles for a video using OpenAI Whisper `large-v3` (excellent on Hebrew).

**Before you run:** menu **Runtime → Change runtime type → GPU (T4) → Save**. Then **Runtime → Run all**.

Produces two files you can download at the end:
- `english.srt` — Hebrew speech translated to English (safe single file for a mixed-language audience).
- `hebrew.srt` — Hebrew captions of what is spoken.

On a free T4 GPU this runs several times faster than real time.

In [ ]:
# 1. Install
!pip -q install -U faster-whisper gdown

In [ ]:
# 2. Fetch the video (pre-filled with your Drive file ID)
FILE_ID = "1xUovmwPhrKb5aW8SYlS4WWMjFJgSXGOP"
!gdown {FILE_ID} -O movie.mp4
import os; print("size MB:", round(os.path.getsize('movie.mp4')/1e6, 1))
MEDIA = "movie.mp4"

# --- Alternatives if gdown fails (e.g. sharing turned off) ---
# Upload instead:
#   from google.colab import files; up = files.upload(); MEDIA = list(up.keys())[0]
# Or mount your Drive and point at the path:
#   from google.colab import drive; drive.mount('/content/drive')
#   MEDIA = '/content/drive/MyDrive/.../videoplayback (1).mp4'

In [ ]:
# 3. Transcribe -> english.srt + hebrew.srt
import torch
from faster_whisper import WhisperModel

use_gpu = torch.cuda.is_available()
print("GPU:", use_gpu, "(if False, set Runtime -> GPU for big speedup)")
model = WhisperModel("large-v3",
                     device="cuda" if use_gpu else "cpu",
                     compute_type="float16" if use_gpu else "int8")

def ts(t):
    h = int(t // 3600); m = int(t % 3600 // 60); s = int(t % 60); ms = int((t - int(t)) * 1000)
    return f"{h:02}:{m:02}:{s:02},{ms:03}"

def run(task, out):
    # transcribe -> Hebrew captions (force he); translate -> English (auto-detect source)
    segs, info = model.transcribe(
        MEDIA, task=task,
        language="he" if task == "transcribe" else None,
        vad_filter=True, beam_size=5)
    with open(out, "w", encoding="utf-8") as f:
        for i, s in enumerate(segs, 1):
            line = s.text.strip()
            f.write(f"{i}\n{ts(s.start)} --> {ts(s.end)}\n{line}\n\n")
            print(ts(s.start), line)
    print("WROTE", out)

run("translate", "english.srt")   # Hebrew speech -> English subtitles
run("transcribe", "hebrew.srt")   # Hebrew speech -> Hebrew captions

In [ ]:
# 4. Download
from google.colab import files
files.download("english.srt")
files.download("hebrew.srt")

## At the screening
Play the video in **VLC** → menu **Subtitle → Add Subtitle File** → pick the `.srt`. Nothing is re-encoded.

Because the clip is short, skim the printed lines and fix any obvious slip by hand — that takes you from *good* to *perfect*.

To burn subtitles in permanently: `ffmpeg -i movie.mp4 -vf subtitles=english.srt out.mp4`.